# Analyze imported BBP simulation campaigns

In [ ]:
import json
from bluepysnap import Circuit, Simulation
from entitysdk import models
from entitysdk.client import Client
from entitysdk.staging.circuit import stage_circuit
from entitysdk.staging.simulation_result import stage_simulation, stage_simulation_result
from obi_auth import get_token
from obi_notebook import get_entities
from obi_notebook.get_environment import get_environment
from obi_notebook.get_projects import get_projects
from pathlib import Path
from tqdm import tqdm

## Campaign selection and download

#### Get authentication token and choose a project context

In [ ]:
token = get_token(environment=get_environment(), auth_mode="daf")
project_context = get_projects(token=token)

#### Initialize the client with the token and project context

In [ ]:
client = Client(environment=get_environment(), project_context=project_context, token_manager=token)

#### Load BBP whisker scan campaign and show details (optional)
Campaign name: `BBP-SM-whisker-scan`

In [ ]:
campaign_name = "BBP-SM-whisker-scan"
show_details = False

# Retrieve campaign entity
res = client.search_entity(entity_type=models.SimulationCampaign, query={"name": campaign_name}).all()
if len(res) == 0:
    print(f"ERROR: BBP whisker scan campaign '{campaign_name}' not found in database!")
elif len(res) > 1:
    print(f"WARNING: BBP whisker scan campaign '{campaign_name}' found multiple times in database - using first instance!")
simulation_campaign = res[0]

# Get campaign information
circuit = client.get_entity(entity_id=simulation_campaign.entity_id, entity_type=models.Circuit)
print(f"Loaded campaign '{simulation_campaign.name}' (ID {simulation_campaign.id}):\n")
print(simulation_campaign.description)
print()
print(f"Circuit: '{circuit.name}' (ID {circuit.id})\n")
print(f"Number of simulations: {len(simulation_campaign.simulations)}\n")
print("Scan parameters:")
print("\n".join([f"  {k}: {v}" for k, v in simulation_campaign.scan_parameters.items()]))

# Show campaign details (optional)
if show_details:
    print("\nList of simulations:")
    for sim in simulation_campaign.simulations:
        sim = client.get_entity(entity_id=sim.id, entity_type=models.Simulation)  # To get assets!!
        print(f"  Simulation '{sim.name}' (ID {sim.id}): ")
        print(f"    {sim.scan_parameters}")

#### Stage the campaign together with the corresponding circuit
If the entities have been staged before, they will be loaded from cache instead of staging them again.

In [ ]:
CIRCUIT_STAGING_ROOT = Path("/home/shared_data/entity_cache/sonata_circuit")
CAMPAIGN_STAGING_ROOT = Path("./staged_simulation_campaigns")

In [ ]:
def stage_campaign_circuit(simulation_campaign_id, output_root):
    """Stage the circuit of a simulation campaign, incl. caching."""
    # Get circuit ID from simulation campaign
    simulation_campaign = client.get_entity(entity_type=models.SimulationCampaign, entity_id=simulation_campaign_id)
    circuit_id = str(simulation_campaign.entity_id)
    circuit_entity = client.get_entity(entity_type=models.Circuit, entity_id=circuit_id)

    # Stage circuit or use from cache
    output_dir = Path(output_root).resolve() / circuit_id
    if output_dir.exists():
        print(f"INFO: Using cached circuit '{circuit_entity.name}'")
        circuit_config_path = output_dir / "circuit_config.json"
    else:
        circuit_config_path = stage_circuit(
            client=client,
            model=circuit_entity,
            output_dir=output_dir,
            max_concurrent=4,
        )
    assert circuit_config_path.is_file(), "ERROR: Circuit staging error - clear cache and retry!"
    print(f"INFO: Staged circuit '{circuit_entity.name}' at {circuit_config_path}")

    return circuit_config_path

def stage_sim_campaign(simulation_campaign_id, output_root, circuit_config_path):
    """Stage the simulation campaign w/o circuit, incl. caching."""
    simulation_campaign = client.get_entity(entity_type=models.SimulationCampaign, entity_id=simulation_campaign_id)
    campaign_output_dir = Path(output_root).resolve() / str(simulation_campaign_id)

    # Stage individual simulations and results
    num_cached = 0
    num_staged = 0
    simulation_config_paths = []
    for simulation in tqdm(simulation_campaign.simulations):
        # IMPORTANT: The order of simulations returned by simulation_campaign.simulations is not necessarily ordered!!

        simulation = client.get_entity(entity_id=simulation.id, entity_type=models.Simulation)  # Required to get assets!!

        # Stage simulation + result or use from cache
        sim_dir = simulation.name.replace(" ", "_").lower()
        output_dir = campaign_output_dir / sim_dir
        if output_dir.exists():
            num_cached += 1
            # print(f"INFO: Using cached simulation '{simulation.name}'")
            simulation_config_path = output_dir / "simulation_config.json"
        else:
            # Stage simulation
            num_staged += 1
            simulation_config_path = stage_simulation(
                client=client,
                model=simulation,
                output_dir=output_dir,
                circuit_config_path=circuit_config_path,
            )

            # Get simulation execution and stage simulation result
            simulation_executions = client.search_entity(
                entity_type=models.SimulationExecution,
                query={"used__id": simulation.id}
            ).all()

            if len(simulation_executions) == 0:
                print(f"WARNING: No SimulationExecution/Result found for Simulation ID {simulation.id}")
            else:
                simulation_execution = simulation_executions[0]
                simulation_result_id = simulation_execution.generated[0].id
                simulation_result = client.get_entity(entity_type=models.SimulationResult, entity_id=simulation_result_id)

                _ = stage_simulation_result(
                    client=client,
                    model=simulation_result,
                    output_dir=output_dir,
                    simulation_config_file=simulation_config_path,
                )
        
        assert simulation_config_path.is_file(), "ERROR: Simulation staging error - clear cache and retry!"
        # print(f"INFO: Staged simulation '{simulation.name}' at {simulation_config_path}")

        simulation_config_paths.append(str(simulation_config_path))

    # Stage campaign summary
    config_path = None
    for asset in simulation_campaign.assets:
        if asset.label == "campaign_summary":
            config_path = client.download_file(
                asset_id=asset.id,
                entity_id=simulation_campaign_id,
                entity_type=models.SimulationCampaign,
                output_path=campaign_output_dir,
            )
            break

    if config_path is not None:
        # Update paths of campaign config
        with open(config_path, "r") as f:
            config_dict = json.load(f)
        config_dict["attrs"]["path_prefix"] = "./"
        config_dict["attrs"]["circuit_config"] = str(staged_circuit_config)
        with open(config_path, "w") as f:
            json.dump(config_dict, f)

    print(f"INFO: Staged campaign '{simulation_campaign.name}' with {len(simulation_config_paths)} simulation(s) ({num_staged} downloaded, {num_cached} cached) at {campaign_output_dir}")

    return simulation_config_paths, config_path


In [ ]:
staged_circuit_config = stage_campaign_circuit(
    simulation_campaign_id=simulation_campaign.id,
    output_root=CIRCUIT_STAGING_ROOT,
)

In [ ]:
staged_sim_configs, staged_campaign_config_path = stage_sim_campaign(
    simulation_campaign_id=simulation_campaign.id,
    output_root=CAMPAIGN_STAGING_ROOT,
    circuit_config_path=staged_circuit_config
)

## BlueETL campaign analysis

BlueETL analyses are defined in a .yaml file which is gernerated from the analysis configuration dict below. Here we will extract spike trains for all value combinations of the parameter scan in a time window around the whisker flick (centered at t = 0 ms) and displays them in an interactive way.

In [ ]:
import logging
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from blueetl.analysis import run_from_file
from ipywidgets import widgets, interact

In [ ]:
# Define analysis config and write to .yaml file
analysis_config_file = "./analysis_config.yaml"
analysis_config_dict = {
    "version": 4,
    "simulation_campaign": str(staged_campaign_config_path),
    "cache": {"path": "./analysis_output", "clear": False},
    "analysis": {
        "hex0_spikes": {
            "extraction": {
                "report": {"type": "spikes"},
                "limit": None,
                "neuron_classes": {
                    "L23_EXC": {"query": {"layer": [2, 3], "synapse_class": ["EXC"]}},
                    "L4_EXC": {"query": {"layer": [4], "synapse_class": ["EXC"]}},
                    "L5_EXC": {"query": {"layer": [5], "synapse_class": ["EXC"]}},
                    "L6_EXC": {"query": {"layer": [6], "synapse_class": ["EXC"]}},
                    "L1_INH": {"query": {"layer": [1], "synapse_class": ["INH"]}},
                    "L23_INH": {"query": {"layer": [2, 3], "synapse_class": ["INH"]}},
                    "L4_INH": {"query": {"layer": [4], "synapse_class": ["INH"]}},
                    "L5_INH": {"query": {"layer": [5], "synapse_class": ["INH"]}},
                    "L6_INH": {"query": {"layer": [6], "synapse_class": ["INH"]}},
                    "L23_PV": {"query": {"layer": [2, 3], "mtype": ["L23_LBC", "L23_NBC", "L23_CHC"]}},
                    "L4_PV": {"query": {"layer": [4], "mtype": ["L4_CHC", "L4_NBC", "L4_LBC"]}},
                    "L5_PV": {"query": {"layer": [5], "mtype": ["L5_CHC", "L5_LBC", "L5_NBC"]}},
                    "L6_PV": {"query": {"layer": [6], "mtype": ["L6_LBC", "L6_CHC", "L6_NBC"]}},
                    "L23_SST": {"query": {"layer": [2, 3], "mtype": ["L23_BTC", "L23_NGC", "L23_SBC", "L23_MC", "L23_DBC"]}},
                    "L4_SST": {"query": {"layer": [4], "mtype": ["L4_NGC", "L4_DBC", "L4_SBC", "L4_BTC", "L4_MC"]}},
                    "L5_SST": {"query": {"layer": [5], "mtype": ["L5_SBC", "L5_BTC", "L5_DBC", "L5_NGC", "L5_MC"]}},
                    "L6_SST": {"query": {"layer": [6], "mtype": ["L6_BTC", "L6_NGC", "L6_MC", "L6_SBC", "L6_DBC"]}},
                    "L23_5HT3aR": {"query": {"layer": [2, 3], "mtype": ["L23_BP"]}},
                    "L4_5HT3aR": {"query": {"layer": [4], "mtype": ["L4_BP"]}},
                    "L5_5HT3aR": {"query": {"layer": [5], "mtype": ["L5_BP"]}},
                    "L6_5HT3aR": {"query": {"layer": [6], "mtype": ["L6_BP"]}},
                    "ALL_INH": {"query": {"synapse_class": ["INH"]}, "limit": None},
                    "ALL_EXC": {"query": {"synapse_class": ["EXC"]}, "limit": None},
                    "ALL": {"limit": None}},
                "windows": {
                    "unconn_2nd_half": {"bounds": [500, 1000], "window_type": "spontaneous"},
                    "conn_spont": {"bounds": [1500, 2500], "window_type": "spontaneous"},
                    "evoked_SOZ_25ms": {"bounds": [0, 25], "initial_offset": 1500, "n_trials": 10, "trial_steps_value": 1000,
                                        "window_type": "evoked_stimulus_onset_zeroed"},
                    "evoked_SOZ_250ms": {"bounds": [-50, 250], "initial_offset": 1500, "n_trials": 10, "trial_steps_value": 1000,
                                         "window_type": "evoked_stimulus_onset_zeroed"}},
                "population": "S1nonbarrel_neurons",
                "node_set": "hex0"}
        }
    }
}

with open(analysis_config_file, "w") as f:
    yaml.dump(analysis_config_dict, f)

In [ ]:
# Run analysis
ma = run_from_file(analysis_config_file, loglevel=logging.ERROR)
ma = ma.apply_filter()

In [ ]:
# Create spike results dataframe merged with simulation conditions
df_spikes = ma.hex0_spikes.repo.report.df
df_sims = ma.hex0_spikes.repo.simulations.df
df_merged = pd.merge(df_spikes, df_sims, on="simulation_id")

In [ ]:
# Interactive plot function
def plot_fct(res_sel_ca, res_sel_R_OU, res_sel_P_FR, res_sel_VPM, res_sel_wnd):
    # Select spike trains
    df_spikes_sel_exc = df_merged.etl.q(neuron_class="ALL_EXC", window=res_sel_wnd, trial=0, ca=res_sel_ca, depol_stdev_mean_ratio=res_sel_R_OU, desired_connected_proportion_of_invivo_frs=res_sel_P_FR, vpm_pct=res_sel_VPM)
    df_spikes_sel_inh = df_merged.etl.q(neuron_class="ALL_INH", window=res_sel_wnd, trial=0, ca=res_sel_ca, depol_stdev_mean_ratio=res_sel_R_OU, desired_connected_proportion_of_invivo_frs=res_sel_P_FR, vpm_pct=res_sel_VPM)
    
    # Plot spike rasters
    plt.figure()
    plt.plot(df_spikes_sel_exc["time"], df_spikes_sel_exc["gid"], ",r", label="EXC")
    plt.plot(df_spikes_sel_inh["time"], df_spikes_sel_inh["gid"], ",b", label="INH")
    plt.ylim(plt.ylim())
    # plt.vlines(0, ymin=min(plt.ylim()), ymax=max(plt.ylim()), colors="k", alpha=0.5, zorder=1e6, label="Flick")
    plt.title(f"Raster plot\n[Ca {res_sel_ca}, R_OU {res_sel_R_OU}, P_FR {res_sel_P_FR}, VPM {res_sel_VPM}%]")
    plt.xlabel("Time (ms)")
    plt.ylabel("Neuron ID")
    plt.legend()
    plt.show()

In [ ]:
res_sel_ca_wdgt = widgets.Dropdown(options=df_sims["ca"].unique(), description="Calcium:", style={"description_width": "auto"}, layout=widgets.Layout(width="max-content"))
res_sel_R_OU_wdgt = widgets.Dropdown(options=df_sims["depol_stdev_mean_ratio"].unique(), description="Ornstein-Uhlenbeck std to mean ratio:", style={"description_width": "auto"}, layout=widgets.Layout(width="max-content"))
res_sel_P_FR_wdgt = widgets.Dropdown(options=df_sims["desired_connected_proportion_of_invivo_frs"].unique(), description="In-vivo firing rate proportion:", style={"description_width": "auto"}, layout=widgets.Layout(width="max-content"))
res_sel_VPM_wdgt = widgets.Dropdown(options=df_sims["vpm_pct"].unique(), description="VPM percentage:", style={"description_width": "auto"}, layout=widgets.Layout(width="max-content"))
res_sel_wnd_wdgt = widgets.Dropdown(options=[w for w in df_merged["window"].unique() if "evoked" in w], description="Time window:", style={"description_width": "auto"}, layout=widgets.Layout(width="max-content"))

iplot = interact(plot_fct, res_sel_ca=res_sel_ca_wdgt, res_sel_R_OU=res_sel_R_OU_wdgt, res_sel_P_FR=res_sel_P_FR_wdgt, res_sel_VPM=res_sel_VPM_wdgt, res_sel_wnd=res_sel_wnd_wdgt)